<a href="https://colab.research.google.com/github/lipeluiz12310/python-lab/blob/main/RodasMexico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

caminho_imagem = 'teste09.jpg'

# Calibração

corte_topo = 0.12
corte_base = 0.05
corte_esquerda = 0.17
corte_direita = 0.17
limiar_preto = 30

# Caixa Delimitadora
area_minima = 1500
area_maxima = 15000
largura_min = 30   # Largura mínima em pixels
largura_max = 300  # Largura máxima em pixels
altura_min = 30    # Altura mínima em pixels
altura_max = 100   # Altura máxima em pixels


img_original = cv2.imread(caminho_imagem)

# Verifica se a imagem existe
if img_original is None:
    raise FileNotFoundError(f"ERRO: A imagem '{caminho_imagem}' não foi encontrada. Faça o upload na pastinha à esquerda!")

h, w, _ = img_original.shape

roi_h_start = int(h * corte_topo)
roi_h_end = int(h * (1 - corte_base))
roi_w_start = int(w * corte_esquerda)
roi_w_end = int(w * (1 - corte_direita))

img_recortada = img_original[roi_h_start:roi_h_end, roi_w_start:roi_w_end]

img_rgb = cv2.cvtColor(img_recortada, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_recortada, cv2.COLOR_BGR2GRAY)

suave = cv2.GaussianBlur(gray, (5, 5), 0)
_, binarizada = cv2.threshold(suave, limiar_preto, 255, cv2.THRESH_BINARY_INV)

tamanho_kernel = 1
max_tentativas = 10
tentativa = 0
rodas_encontradas = 5

while rodas_encontradas > 4 and tentativa < max_tentativas:
    kernel = np.ones((tamanho_kernel, tamanho_kernel), np.uint8)
    limpa = cv2.morphologyEx(binarizada, cv2.MORPH_CLOSE, kernel)
    contornos, _ = cv2.findContours(limpa, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    img_resultado = img_rgb.copy()
    rodas_encontradas = 0

    print(f"\n--- Tentativa {tentativa + 1} (Kernel {tamanho_kernel}x{tamanho_kernel}) ---")

    for contorno in contornos:
        area = cv2.contourArea(contorno)
        x, y, w_box, h_box = cv2.boundingRect(contorno)

        # Descomente a linha abaixo para ver o tamanho de TUDO o que o código acha
        # print(f"Área: {area:.0f} | Largura: {w_box} | Altura: {h_box}")

        # Tem que passar na Área E na Largura E na Altura
        if area_minima < area < area_maxima:
            if (largura_min < w_box < largura_max) and (altura_min < h_box < altura_max):
                cv2.rectangle(img_resultado, (x, y), (x + w_box, y + h_box), (0, 255, 0), 3)
                rodas_encontradas += 1

    if rodas_encontradas <= 4:
        break

    tamanho_kernel += 2
    tentativa += 1

if tentativa > 0 and tamanho_kernel > 1:
    print(f"\n[*] AUTO-CALIBRAÇÃO ATIVADA: O código ajustou o kernel para {tamanho_kernel}x{tamanho_kernel} sozinho.")

status = "OK" if rodas_encontradas == 4 else "Reprovado"
defeito = "Rodas Faltando" if rodas_encontradas < 4 else "Nenhum"
if rodas_encontradas > 4: defeito = "Erro Crítico: Ruído não resolvido"

dados = {
    'Timestamp': [datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
    'ID_Peca': ['Carrinho_001'],
    'Rodas_Contadas': [rodas_encontradas],
    'Status': [status],
    'Tipo_de_Falha': [defeito]
}

df = pd.DataFrame(dados)
print("\n")
print("Dado salvo no sistema")
print(df.to_string(index=False))
print("\n")

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(img_rgb)
plt.title('Imagem analisada')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(limpa, cmap='gray')
plt.title(f'2. Binarização (Kernel {tamanho_kernel}x{tamanho_kernel})')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(img_resultado)
plt.title(f'3. Resultado: {rodas_encontradas} Rodas')
plt.axis('off')

plt.tight_layout()
plt.show()

FileNotFoundError: ERRO: A imagem 'teste09.jpg' não foi encontrada. Faça o upload na pastinha à esquerda!